# Curves: canonical divisors, point counts and zeta functions

## Setup and canonical divisors

In [1]:
implicit_multiplication(True)

In [2]:
def get_canonical(C):
    return C.canonical_divisor()

def get_degree_canonical(C):
    K = get_canonical(C)
    canonical_map = C.complete_linear_system(K).morphism_to_projective_space()
    D = canonical_map.image()
    return -1 * D.degree()

def genus_degree_formula(X):
    return _genus_degree_formula(
        X.defining_polynomial().degree(),
        X.singular_points()
    )

def _genus_degree_formula(d, rs):
    return (1/2)*(d-1)*(d-2) - 1/2*sum([r * r-1 for r in rs])

def irr(C):
    return C.arithmetic_genus() - C.geometric_genus()

def proj_closure(Y):
    return Y.projective_closure()

def summarize(Y, xbounds=(-5, 5), ybounds=(-5, 5)):
    f = Y.defining_polynomial()
    d = f.degree()
    X = proj_closure(Y)
    X_sing = X.singular_points()
    mults = [X.multiplicity(p) for p in X_sing]
    pa = X.arithmetic_genus()
    pg = X.geometric_genus()

    if X.is_smooth():
        assert pa == pg
    if X.is_irreducible() and all([X.is_ordinary_singularity(p) for p in X_sing]):
        assert pa == _genus_degree_formula(d, []), f"p_a = {pa}, formula yields {_genus_degree_formula(d, [])}"
        assert pg == _genus_degree_formula(d, mults), f"p_g = {pg}, formula yields {_genus_degree_formula(d, mults)} using {mults}"

    K = get_canonical(X)
    dK = K.degree()
    assert dK == 2*pg - 2

    print(f"""
Polynomial: {f}
Homogenization: {X.defining_polynomial()}
Degree: {d}
Smoothness: Affine: {Y.is_smooth()}, Projective: {X.is_smooth()}
Irreducible? {X.is_irreducible()}
Singular points: {X_sing}
Multiplicities: {mults}
Geometric genus: {pg}
Arithmetic genus: {pa}
Irregularity: {irr(X)}
K_X: {K}
deg(K_X): {dK}
    """)
    if pg == 1:
        E = X.jacobian()
        print(f"Equivalent to: {E}")
        jE = E.j_invariant()
        print(f"j={jE}")
        assert X.j_invariant() == jE
        p = next(p for p in E.good_reduction_primes())
        Ep = E.reduction(p)
        T = ZZ.power_series_ring("t")
        Zt = Ep.zeta_function().power_series(T, precision=3)
        print(f"Zeta series for E over F_{{{p}}}: {Zt}")
    print(f"Point counts over F_{{3^k}}: {get_poly(X, p=3, n=5)}")
    return Y.real_locus_plot(x_range=xbounds, y_range=ybounds, gridlines=True, figsize=[10, 10])

def get_poly(C, p=2, n=7):
    Zt = ZZ.polynomial_ring("t")
    t = Zt.algebra_generator("t")
    C_p = C.reduction(p)
    return sum([C_p.point_count(k) * t^k for k in range(1, n + 1)])

## Example curves

In [3]:
plane = ProjectiveSpaces(GF(2))(2, names=("x", "y", "z"))
x, y, z = plane.homogeneous_coordinate_ring().algebra_generators()
f = x^3*y + y^3*z + x*z^3
C = plane.closed_subscheme(f); pts = list(C.rational_points())
D = 4*C.divisor(pts[0]) + 4*C.divisor(pts[2])
D

AttributeError: 'OwnedJoinCategory.parent_class' object has no attribute 'rational_points'

In [4]:
P = QQ.polynomial_ring(("x", "y"))
x, y = P.algebra_generators()
X = Curves(QQ).from_equation(y^2 - 2*x^5 - 2*x^3 + 3*x - 2)
summarize(X)

AttributeError: 'OwnedJoinCategory.parent_class' object has no attribute 'defining_polynomial'

## Fitting quartic curves through four points

In [5]:
Q = QQ.polynomial_ring(("a", "b", "c", "x", "y"))
a, b, c, x, y = Q.algebra_generators()

points = [(1, 2), (1, -2), (-2, 1), (-2, -1)]

f = y^2 - a*x^4 - b*x^2 + c*x - 1

eqns = [f.substitute({x: p[0], y: p[1]}) for p in points]
solns = Q.ideal(eqns).affine_solution_set([a, b, c])
print(solns)

AttributeError: 'OwnedJoinCategory.parent_class[_PresentedModule].ObjectType' object has no attribute 'affine_solution_set'

In [6]:
assert solns.cardinality() > 0, "No solutions."
soln = solns.an_element()
print(soln)
P = QQ.polynomial_ring(("x", "y"))
F = P(f.substitute(soln).substitute({c: 2}))
print(F)
X = Curves(QQ).from_equation(F)
E = X.projective_closure().jacobian()
summarize(X, xbounds=(-4, 3), ybounds=(-4, 4))

NameError: name 'solns' is not defined

In [7]:
# 2*x^4 - 7*x^2 + y^2 + 2*x - 1
g = y^2 - a*x^4 - b*x^2 + 2*x - 1
points = [(1, 2), (1, -2), (-2, 1), (-2, -1)]

Q.ideal([g.substitute({x: p[0], y: p[1]}) for p in points]).affine_solution_set([a, b])

AttributeError: 'OwnedJoinCategory.parent_class[_PresentedModule].ObjectType' object has no attribute 'affine_solution_set'

In [8]:
[g.substitute({x: p[0], y: p[1]}) for p in points]

[-a*x^4 - b*x^2 + y^2 + 2*x - 1,
 -a*x^4 - b*x^2 + y^2 + 2*x - 1,
 -a*x^4 - b*x^2 + y^2 + 2*x - 1,
 -a*x^4 - b*x^2 + y^2 + 2*x - 1]

In [9]:
M = ZZ.matrix_space(2, 2).from_rows([[-1, -1], [-16, -4]])
rhs = M.codomain()([-5, 4])

print(M, rhs)
a1, b1 = M.solve_right(rhs)
print(a1, b1)
g.substitute({a: a1, b: b1})

Generic endomorphism of Free module on {[0], [1]} over Integer Ring -5*[0] + 4*[1]


-2 7


-a*x^4 - b*x^2 + y^2 + 2*x - 1

In [10]:
P = QQ.polynomial_ring(("x", "y"))
x, y = P.algebra_generators()

h = y^2 - a1*x^4 - 1*x^2 + 2*x - 1
print(h)

X = Curves(QQ).from_equation(h)
summarize(X)

2*x^4 - x^2 + y^2 + 2*x - 1


AttributeError: 'OwnedJoinCategory.parent_class' object has no attribute 'defining_polynomial'

## Scratch computations

In [11]:
A = ZZ.polynomial_ring("t")
t = A.algebra_generator("t")
Z = A.zero()

for k in range(1, 6):
    F = GF(3^k)
    plane = AffineSpaces(F)(2, names=("x", "y"))
    x, y = plane.coordinate_ring().algebra_generators()
    h = 2*x^4 - x^2 + y^2 + 2*x - 1
    Z = Z + plane.closed_subscheme(h).point_count() * t^k

print(Z)

AttributeError: 'AffineScheme_with_category' object has no attribute 'rational_points'

In [12]:
K = QuadraticField(-1, "i")
i = K.algebra_generator("i")
Kl = K.polynomial_ring("l")
l = Kl.algebra_generator("l")
conjugation = [sigma for sigma in i.conjugates(K).keys() if i.conjugates(K)[sigma] == -i][0]
a = 2 + i*l
a^2 + a.map_coefficients(conjugation)^2

AttributeError: 'Owned_NativeFreeAlgebraParent_with_category.element_class' object has no attribute 'map_coefficients'

In [13]:
(a^2 + a.map_coefficients(conjugation)^2 - 1).roots(RR)

AttributeError: 'Owned_NativeFreeAlgebraParent_with_category.element_class' object has no attribute 'map_coefficients'

In [14]:
a.norm()

TypeError: norm() takes exactly 1 positional argument (0 given)

In [15]:
NS[1]
n_elts(1)
#NS[2]
#n_elts(2)
a.norm()

NameError: name 'NS' is not defined

## Point counts and zeta functions

In [16]:
T = ZZ.power_series_ring("t")
t = T.power_series_variable()

q = 17

def n_elts(alpha, m):
    return (q^m + 1) - (alpha^m + alpha.conjugate()^m)


P = ZZ.polynomial_ring(("x", "y"))
x, y = P.algebra_generators()

X = Curves(ZZ).from_equation(2*x^4 - x^2 + y^2 + 2*x - 1)
assert X.geometric_genus() == 1

Xq = X.reduction(q)
assert Xq.geometric_genus() == 1
assert Xq.is_smooth()


PXq = Xq.projective_closure()
assert PXq.geometric_genus() == 1
assert PXq.is_smooth()


point_counts = [PXq.point_count(m) for m in range(1, 5)]
display(f"{[Xq.point_count(m) for m in range(1, 5)]} -> {point_counts}")

frobenius_eigenvalues = PXq.zeta_function().numerator().reverse().roots(QQbar)
display(frobenius_eigenvalues)
alpha = frobenius_eigenvalues[0][0]
assert point_counts[0] == n_elts(alpha, 1)
assert point_counts[1] == n_elts(alpha, 2)
print(f"alpha = {alpha}. Norm = {alpha.abs()}")
assert alpha.abs() == sqrt(q)

ValueError: Coefficient ring must be a field for function 'dimension'.

In [17]:
PX5 = X5.projective_closure()
PX5.point_count

NameError: name 'X5' is not defined

In [18]:
[n_elts(alpha, m) for m in [1, 2, 3, 4]]

NameError: name 'alpha' is not defined

In [19]:
PX5.geometric_genus()

NameError: name 'PX5' is not defined

In [20]:
Zt = X.reduction(3).zeta_function().power_series(T, precision=4)

Zpt = t * Zt.log().derivative()
Zpt

AttributeError: 'OwnedJoinCategory.parent_class' object has no attribute 'reduction'

In [21]:
q = 5
qms = [q^m + 1 for m in range(1, 6)]
qms

[6, 26, 126, 626, 3126]

## Jacobian of a genus-one curve

In [22]:
f = X.defining_polynomial()  # the genus-1 polynomial
E = X.projective_closure().jacobian()
E

AttributeError: 'OwnedJoinCategory.parent_class' object has no attribute 'defining_polynomial'

In [23]:
Fp = GF(5)
E_fp = E.reduction(5)

AttributeError: 'function' object has no attribute 'reduction'

In [24]:
(i*y)^2 + (-i*y)^2

TypeError: Unable to coerce i to an integer